In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import glob, os, warnings
warnings.filterwarnings('ignore')

data_folder = r'C:\Users\paulc\OneDrive\Projects\miami-open-predictor\data'

def load_tennis_data_file(filepath):
    if filepath.endswith('.csv'):
        df = pd.read_csv(filepath)
    else:
        df = pd.read_excel(filepath)
    df = df.rename(columns={
        'Tournament': 'tourney_name', 'Date': 'tourney_date',
        'Surface': 'surface', 'Round': 'round',
        'Winner': 'winner_name', 'Loser': 'loser_name',
        'WRank': 'winner_rank', 'LRank': 'loser_rank', 'Best of': 'best_of'
    })
    keep = ['tourney_name','tourney_date','surface','round',
            'winner_name','loser_name','winner_rank','loser_rank','best_of']
    df = df[[c for c in keep if c in df.columns]]
    df['tourney_date'] = pd.to_datetime(df['tourney_date'], errors='coerce')
    df['source'] = 'tennis_data'
    return df

excel_files = glob.glob(os.path.join(data_folder, '*.xlsx')) + \
              glob.glob(os.path.join(data_folder, '*.xls')) + \
              glob.glob(os.path.join(data_folder, '2025*.csv'))

td_dfs = [load_tennis_data_file(f) for f in sorted(excel_files)]
td_combined = pd.concat(td_dfs, ignore_index=True)

# Load 2024 Sackmann
df_2024 = pd.read_csv(os.path.join(data_folder, 'atp_matches_2024.csv'))
df_2024['tourney_date'] = pd.to_datetime(df_2024['tourney_date'].astype(str), format='%Y%m%d', errors='coerce')
df_2024['best_of'] = df_2024.get('best_of', 3)
keep = ['tourney_name','tourney_date','surface','round','winner_name','loser_name','winner_rank','loser_rank','best_of']
df_2024 = df_2024[[c for c in keep if c in df_2024.columns]]
df_2024['source'] = 'sackmann'

all_matches = pd.concat([td_combined, df_2024], ignore_index=True)

for col in ['surface','winner_name','loser_name','tourney_name','round']:
    all_matches[col] = all_matches[col].astype(str).str.strip().str.lower()

all_matches = all_matches.dropna(subset=['tourney_date','surface','winner_name','loser_name','winner_rank','loser_rank'])
all_matches['winner_rank'] = pd.to_numeric(all_matches['winner_rank'], errors='coerce')
all_matches['loser_rank']  = pd.to_numeric(all_matches['loser_rank'], errors='coerce')
all_matches = all_matches.dropna(subset=['winner_rank','loser_rank'])
all_matches['year'] = all_matches['tourney_date'].dt.year
all_matches = all_matches.sort_values('tourney_date').reset_index(drop=True)

print(f"Total matches: {len(all_matches)}")
print(f"Surfaces: {all_matches['surface'].unique()}")

Total matches: 67652
Surfaces: ['hard' 'clay' 'grass' 'carpet']


In [2]:
hard = all_matches[all_matches['surface'] == 'hard'].copy().reset_index(drop=True)
print(f"Hard court matches: {len(hard)}")
print(f"Miami Open matches: {hard[hard['tourney_name'].str.contains('miami', na=False)].shape[0]}")

hard.to_csv(os.path.join(data_folder, 'hard_combined_2000_2025.csv'), index=False)
print("Saved!")

Hard court matches: 36205
Miami Open matches: 475
Saved!


In [3]:
def compute_features(df_input):
    df = df_input.copy().sort_values('tourney_date').reset_index(drop=True)

    for col in ['winner_elo','loser_elo','winner_hard_win_pct','loser_hard_win_pct',
                'winner_form5_hard','loser_form5_hard','winner_form10_all',
                'loser_form10_all','winner_last_played','loser_last_played']:
        df[col] = np.nan

    elo         = defaultdict(lambda: 1500.0)
    hard_wins   = defaultdict(int)
    hard_games  = defaultdict(int)
    form_hard   = defaultdict(list)
    form_all    = defaultdict(list)
    last_played = defaultdict(lambda: pd.NaT)
    K = 32

    for i, row in df.iterrows():
        w, l = row['winner_name'], row['loser_name']
        date = row['tourney_date']

        df.at[i, 'winner_elo']           = elo[w]
        df.at[i, 'loser_elo']            = elo[l]
        df.at[i, 'winner_hard_win_pct']  = hard_wins[w] / hard_games[w] if hard_games[w] else 0.0
        df.at[i, 'loser_hard_win_pct']   = hard_wins[l] / hard_games[l] if hard_games[l] else 0.0
        df.at[i, 'winner_form5_hard']    = np.mean(form_hard[w][-5:]) if form_hard[w] else 0.0
        df.at[i, 'loser_form5_hard']     = np.mean(form_hard[l][-5:]) if form_hard[l] else 0.0
        df.at[i, 'winner_form10_all']    = np.mean(form_all[w][-10:]) if form_all[w] else 0.0
        df.at[i, 'loser_form10_all']     = np.mean(form_all[l][-10:]) if form_all[l] else 0.0
        df.at[i, 'winner_last_played']   = last_played[w]
        df.at[i, 'loser_last_played']    = last_played[l]

        Rw, Rl = elo[w], elo[l]
        Ew = 1 / (1 + 10**((Rl - Rw) / 400))
        elo[w] += K * (1 - Ew)
        elo[l] += K * (0 - (1 - Ew))

        hard_wins[w]  += 1
        hard_games[w] += 1
        hard_games[l] += 1
        form_hard[w].append(1); form_hard[l].append(0)
        form_all[w].append(1);  form_all[l].append(0)
        last_played[w] = date;  last_played[l] = date

    return df

print("Running feature engineering...")
df_feat = compute_features(hard)
print(f"Done: {df_feat.shape}")

Running feature engineering...
Done: (36205, 21)


In [4]:
df_feat['winner_last_played'] = pd.to_datetime(df_feat['winner_last_played'], errors='coerce')
df_feat['loser_last_played']  = pd.to_datetime(df_feat['loser_last_played'], errors='coerce')

df_feat['rank_diff']             = df_feat['loser_rank'] - df_feat['winner_rank']
df_feat['elo_diff']              = df_feat['winner_elo'] - df_feat['loser_elo']
df_feat['hard_win_pct_diff']     = df_feat['winner_hard_win_pct'] - df_feat['loser_hard_win_pct']
df_feat['form5_hard_diff']       = df_feat['winner_form5_hard'] - df_feat['loser_form5_hard']
df_feat['form10_all_diff']       = df_feat['winner_form10_all'] - df_feat['loser_form10_all']
df_feat['rest_days_winner']      = (df_feat['tourney_date'] - df_feat['winner_last_played']).dt.days.fillna(0)
df_feat['rest_days_loser']       = (df_feat['tourney_date'] - df_feat['loser_last_played']).dt.days.fillna(0)
df_feat['rest_days_diff']        = df_feat['rest_days_winner'] - df_feat['rest_days_loser']
df_feat['month']                 = df_feat['tourney_date'].dt.month
df_feat['month_sin']             = np.sin(2 * np.pi * df_feat['month'] / 12)
df_feat['month_cos']             = np.cos(2 * np.pi * df_feat['month'] / 12)
df_feat['is_miami']              = df_feat['tourney_name'].str.contains('miami', na=False).astype(int)
round_map = {'r128':1,'r64':2,'r32':3,'r16':4,'qf':5,'sf':6,'f':7,'final':7}
df_feat['round_encoded']         = df_feat['round'].map(round_map).fillna(0).astype(int)
df_feat['year']                  = df_feat['tourney_date'].dt.year

feature_cols = [
    'rank_diff', 'elo_diff', 'hard_win_pct_diff',
    'form5_hard_diff', 'form10_all_diff', 'rest_days_diff',
    'month_sin', 'month_cos', 'is_miami', 'round_encoded'
]

df_feat[feature_cols] = df_feat[feature_cols].fillna(0)

# Build balanced dataset
wins   = df_feat.copy(); wins['label'] = 1
losses = df_feat.copy(); losses['label'] = 0
for col in ['rank_diff','elo_diff','hard_win_pct_diff','form5_hard_diff','form10_all_diff','rest_days_diff']:
    losses[col] = -losses[col]

balanced = pd.concat([wins, losses], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
balanced[feature_cols + ['label','year','tourney_name','winner_name','loser_name']].to_csv(
    os.path.join(data_folder, 'hard_features_balanced.csv'), index=False)

print(f"Balanced dataset: {balanced.shape}")
print(f"Label split:\n{balanced['label'].value_counts()}")

Balanced dataset: (72410, 35)
Label split:
label
1    36205
0    36205
Name: count, dtype: int64
